# Stacking statt Abstimmung

**Autor:** jd · **Datum:** 2026-10-29 · **Issue:** –

**Hypothese:** Bei 282 von 482 SVM-Fehlern liegt ein anderes Modell richtig, die Abstimmung (#113) nutzt das nicht: Ein Modell, das als einziges recht hat, unterliegt. Beim Stacking lernt eine logistische Regression aus den Klassen-Scores der Modelle (SVM, Extra Trees, logistische Regression, kNN; Scores aus innerer CV), *wann* sie welchem Modell trauen kann.

**Ablauf:** Entschieden wird nur per CV gegen die Referenz (`combined_svm_aez_month`, gleiches Raster); validiert wird nur, wenn der Gewinn `MIN_CV_GAIN` (0.005) erreicht.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from awp2.config import (
    CROP_COL,
    EARLY_STAGES,
    MIN_CV_GAIN,
    STAGE_COL,
)
from awp2.data import load_split
from awp2.evaluation import compare_predictions, recall_by_class
from awp2.experiment import RunConfig, TuneConfig, out_of_fold_predictions, run, tune
from awp2.models.combined import make_combined_svm
from awp2.models.ensemble import make_combined_stacking
from awp2.plots import plot_cv_scores, plot_metric_comparison, plot_recall_comparison
from awp2.preprocessing import PreprocessingConfig

# A · Läufe

In [ ]:
SLUG = "combined_stacking"
TOPIC = "Stacking"
# reference: combined_svm_aez_month (d93cea89)
REFERENCE = PreprocessingConfig(use_spectral_standard_scale=True, use_aez_month=True)
GRID = {"estimator__C": [100.0, 300.0, 1000.0], "estimator__gamma": [0.001, 0.003, 0.01]}
y_train = load_split().y_train
reference = tune(
    make_combined_svm(),
    TuneConfig(
        name=f"{SLUG}_reference_search",
        approach="combined",
        description="Referenz-SVM, gleiches Raster.",
        param_grid=GRID,
        preprocessing=REFERENCE,
    ),
)
reference_cv = reference.best_score
print(f"Referenz CV: {reference_cv:.4f}")

### Stacking: Regularisierung des Meta-Modells

Zwei Besetzungen: alle vier Modelle und nur SVM + Extra Trees (die beiden stärksten). Die Mitglieder behalten ihre getunten Einstellungen; gesucht wird nur `C` der logistischen Regression oben drauf.

In [ ]:
META_GRID = {"estimator__final_estimator__C": [0.1, 1.0, 10.0]}
candidates = {
    label: tune(
        make_combined_stacking(members),
        TuneConfig(
            name=f"{SLUG}_{key}_search",
            approach="combined",
            description=f"Stacking {label}, Meta-Modell logistische Regression.",
            param_grid=META_GRID,
            preprocessing=REFERENCE,
        ),
    )
    for key, (label, members) in {
        "all": ("SVM + ET + LogReg + kNN", ("svm", "extra_trees", "logreg", "knn")),
        "svm_et": ("SVM + ET", ("svm", "extra_trees")),
    }.items()
}

In [ ]:
searches = {"Referenz": reference, **candidates}
pd.DataFrame(
    {label: {"CV": result.best_score, **result.best_params} for label, result in searches.items()}
).T

# B · Erkenntnisse

## 1. CV-Score je Variante

In [ ]:
ax = plot_cv_scores(
    {label: result.candidates for label, result in searches.items()}, reference=reference_cv
)
ax.set_title(f"{TOPIC}: bester CV-Score je Variante (gestrichelt: Referenz)")

## 2. Gepaart auf dem Trainingsteil (out-of-fold)

Wie viele Zeilen trifft nur die Referenz, wie viele nur das Stacking?

In [ ]:
best_label = max(candidates, key=lambda label: candidates[label].best_score)
best = candidates[best_label]
oof = {
    "Referenz": out_of_fold_predictions(reference.best_model, reference.config),
    best_label: out_of_fold_predictions(best.best_model, best.config),
}
print(f"Beste Variante: {best_label}")
compare_predictions(y_train, oof["Referenz"], oof[best_label])

## 3. Zielfehler

In [ ]:
early = y_train[STAGE_COL].isin(EARLY_STAGES)


def target_errors(pred: pd.DataFrame) -> pd.Series:
    swapped = (
        early
        & y_train[CROP_COL].isin(["corn", "soybean"])
        & pred[CROP_COL].isin(["corn", "soybean"])
        & (y_train[CROP_COL] != pred[CROP_COL])
    )
    return pd.Series(
        {
            "früher Mais ↔ Soja": swapped.sum(),
            "frühe Zeilen, Kultur falsch": (early & (y_train[CROP_COL] != pred[CROP_COL])).sum(),
            "alle Zeilen, Paar falsch": ((y_train != pred).any(axis=1)).sum(),
        }
    )


pd.DataFrame({label: target_errors(pred) for label, pred in oof.items()})

## 4. Recall je Klasse: wo ändert sich etwas?

In [ ]:
plot_recall_comparison(
    {label: recall_by_class(y_train, pred) for label, pred in oof.items()}, top=10
)

## 5. Entscheidung

In [ ]:
gain = best.best_score - reference_cv
print(f"CV-Gewinn {gain:+.4f} (Schwelle {MIN_CV_GAIN})")
if gain >= MIN_CV_GAIN:
    validated = run(
        best.best_model,
        RunConfig(
            name=SLUG,
            approach="combined",
            description=f"{TOPIC}: {best_label}, einmal auf der Validierung.",
            preprocessing=REFERENCE,
            tuning_run=best.run_id,
        ),
    )
    reference_val = run(
        reference.best_model,
        RunConfig(name="reference", approach="combined", preprocessing=REFERENCE, track=False),
    )
    display(
        plot_metric_comparison({"Referenz": reference_val.metrics, best_label: validated.metrics})
    )
else:
    print("Kein Gewinn über der Rauschschwelle – keine Validierung, Referenz bleibt.")

# Fazit

| | CV | Zeilen falsch (out-of-fold) | früher Mais ↔ Soja |
| --- | --- | --- | --- |
| Referenz-SVM | **0.864** | 462 | 64 |
| Stacking SVM + ET + LogReg + kNN | 0.860 | 535 | 74 |
| Stacking SVM + ET | 0.856 | – | – |

- **Hypothese nicht gestützt.** Stacking verliert 0.004 CV-BAcc und trifft auf denselben Zeilen
  deutlich weniger: 130 Zeilen nur die Referenz richtig, 57 nur das Stacking (p < 0.001).
- Das Meta-Modell (`class_weight="balanced"`, stärkste Regularisierung `C=0.1` gewinnt) verschiebt
  Entscheidungen zu seltenen Klassen und verliert dabei bei den häufigen; die Zusatzinformation der
  anderen Modelle kommt auch gemeinsam nicht über die SVM hinaus.
- Keine Validierung; die Referenz bleibt.